# Week 3 · The footprint planner: measure first, then cut where a tonne costs least

**Sentence of the day:** *A footprint is a sum of "amount × emission factor". Find the biggest stage first, then cut where a tonne costs least — not where it looks greenest.*

**What you will build today:** the calculations from this morning's case and this afternoon's card game, in Python. At the end you have three charts that Aylin could show Frau Lindner: the footprint of one roll next to the Frito-Lay chips, the tonnes per year by stage, and a **cost curve** that shows which cut is cheapest.

**How this notebook works**

- Google Colab is a notebook: a document where text and runnable code sit side by side. You run a code cell with the ▶ button on its left, or with **Shift + Enter**.
- Run the cells **from top to bottom**. A cell can only use what the cells above it have created.
- **Part 1** is complete: we run it together. In **Part 2** and **Part 3** your team completes the code. Cells marked **✏️ YOUR TURN** contain blanks `___` (three underscores): replace each blank with your code.
- Every step ends with a **check cell**. It prints ✅ when your result matches the hand calculation, and ❌ with a hint when it does not. As long as a blank is left, the check shows ❌.
- You may use an AI assistant. Check its numbers with the check cells and by hand (Part 4, E3, shows how).

| Part | What | Time |
|---|---|---|
| 0 | Setup | 1 min |
| 1 | Worked example: Aylin's roll, five stages (complete) | 10 min, together |
| 2 | Practice: vans, Blue Star, crates and cups, revenue retrieval | 15 min · split the team: P1 + P2 and P3 + P4 |
| 3 | Planning: the cost curve and the cheapest-first plan | 15 min |
| 4 | Optional extensions for quick teams | if time |

## Part 0 · Setup

This cell loads five **libraries**. A library is a toolbox other people have written, so we do not have to: `pandas` for tables, `numpy` for long lists of numbers, `matplotlib` for charts, `ipywidgets` for sliders, `math` for rounding up. All five are already installed in Colab.

The cell also sets our colours and defines `check`, the helper that prints ✅ or ❌. Run it once.

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets

# Our colours: one per stage of the footprint, plus grey, red and dark grey for the other charts
STAGES = ["Farming", "Manufacture", "Packaging", "Shipping", "Disposal"]       # the five stages, as in the Frito-Lay figure
STAGE_COLOURS = {"Farming": "#7a8f3c", "Manufacture": "#5b6770", "Packaging": "#5aa0c8",
                 "Shipping": "#e09a3e", "Disposal": "#a5524b"}
GREY, RED, DARK, GREEN, ORANGE = "#b5b5b5", "#c1002a", "#1a1a1a", "#2e7d4f", "#e09a3e"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

# Colab normally keeps old outputs under the name ___ . We switch that off, so a blank ___ that
# you have not filled in stops the cell with a clear message instead of using an old output.
try:
    get_ipython().displayhook.cache_size = 0
    get_ipython().user_ns.pop("___", None)
except NameError:
    pass                                   # not running in a notebook: nothing to switch off


def _same(got, expected, tol):
    """True if got equals expected (numbers: within tol; text: exactly; lists: item by item)."""
    if isinstance(expected, (list, tuple)):
        return len(list(got)) == len(expected) and all(_same(g, e, tol) for g, e in zip(list(got), expected))
    if isinstance(expected, str):
        return str(got) == expected
    return abs(float(got) - float(expected)) <= tol


def _show(x):
    """Numbers with thousands separators, so they are easy to read."""
    if isinstance(x, (list, tuple)):
        return "[" + ", ".join(_show(i) for i in x) + "]"
    if isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_)):
        return f"{x:,.2f}".rstrip("0").rstrip(".") if float(x) != int(x) else f"{int(x):,}"
    return str(x)


def check(label, got, expected, tol=0.01, hint="", accept=()):
    """Compare your result with the hand calculation and print ✅ or ❌. `accept` = other correct answers."""
    try:
        ok = any(_same(got, e, tol) for e in [expected, *accept])
    except Exception:
        ok = False
    also = "".join(f" or {_show(a)}" for a in accept)
    print(("✅ " if ok else "❌ ") + f"{label}: you have {_show(got)}, expected {_show(expected)}{also}"
          + ("" if ok else f"   Hint: {hint}"))


def not_yet(error):
    """What a check prints when the cell above still has a blank or an error."""
    if "___" in str(error) or isinstance(error, NameError):
        print("❌ Not done yet: the cell above has not run to its end. Replace every ___ there, "
              "run it, then run this check again.")
    else:
        print("❌ Not working yet:", error)


print("Setup done.")

## Part 1 · Worked example: Aylin's roll, five stages

Frau Lindner wants the carbon footprint per roll, farm to store, in five stages like the Frito-Lay chips. Lena said: *"First we should measure."* Here is her measurement, in Python.

**Carbon footprint** = all greenhouse gases a product causes, directly and indirectly, in g or kg **CO₂e** (all greenhouse gases counted as if they were CO₂).

**Footprint = Σ amount × emission factor.** Σ means "add up". The amount must be in the factor's unit: kWh, litres, kg or tonne-km.

### 1a · The activities of one roll as a table

We store the facts as a **DataFrame**: a table in Python, with named columns, like a small spreadsheet. One row per **activity** (something that causes emissions). The numbers come from Exhibit A (amounts) and Exhibit B (emission factors).

- `amount` = how much of the activity **one delivered roll** needs, in the factor's unit.
- `factor` = kg CO₂e per unit (per kg, per kWh, per litre, per tonne-km).
- `stage` = which of the five stages the activity belongs to.
- `source` = where the factor comes from. ASSUMED = a teaching assumption, not a measured value.

A **variable** is a name for a value: `ROLLS_PER_NIGHT = 1800` stores 1,800 under that name. Every amount is written as the small calculation it comes from, so you can follow it.

In [ ]:
ROLLS_PER_NIGHT = 1800          # rolls on the van every night (Exhibit A)
ROLLS_PER_YEAR = 540_000        # 1,800 × 6 nights × 50 weeks (the underscore is only a thousands separator)
RETURNS = 0.06                  # FrischMarkt throws away 6 % of the delivered rolls unsold
ROLL_G = 60                     # one baked roll weighs 60 g

activities = pd.DataFrame({
    "activity": ["Wheat flour", "Other ingredients", "Bakery electricity",
                 "Plastic bag (LDPE)", "Van diesel", "Flour truck"],
    "stage":    ["Farming", "Farming", "Manufacture", "Packaging", "Shipping", "Shipping"],
    "amount":   [40 / 1000,                        # 40 g flour = 0.040 kg
                 1,                                # the factor is already "per roll"
                 0.035,                            # kWh per roll: mixer, proofer, rack oven
                 25 / 1000 / 20,                   # one 25 g bag (0.025 kg) holds 20 rolls
                 60 * 10 / 100 / ROLLS_PER_NIGHT,  # 60 km × 10 litres per 100 km = 6 litres per night, shared by 1,800 rolls
                 40 / 1_000_000 * 150],            # 40 g = 0.000040 tonnes, moved 150 km = tonne-km
    "unit":     ["kg", "roll", "kWh", "kg", "litre", "tonne-km"],
    "factor":   [0.60, 0.004, 0.363, 2.959, 2.513, 0.05789],     # kg CO2e per unit (Exhibit B)
    "source":   ["ASSUMED (bread LCA)", "ASSUMED", "UBA 2025", "DESNZ 2024", "DESNZ 2024", "DESNZ 2024"],
}).set_index("activity")      # use the activity name as the row label

activities

### 1b · Amount × factor, row by row

One line of code multiplies two **columns** (a column = one named list of values in the table) row by row. We multiply by 1,000 to turn kg into g, because one roll causes only grams.

Then `groupby("stage").sum()` puts the rows with the same stage together and adds them up, like a pivot table in Excel.

In [ ]:
activities["g_co2e"] = activities["amount"] * activities["factor"] * 1000    # g CO2e per delivered roll

print(activities[["stage", "g_co2e"]].round(2))
print()
print("Per stage (g CO2e per delivered roll):")
print(activities.groupby("stage")["g_co2e"].sum().round(2))
print()
print(f"Total: {activities['g_co2e'].sum():.2f} g CO2e per delivered roll")

### 1c · From a delivered roll to a sold roll

FrischMarkt throws away 6 % of the rolls. They were baked, packed and driven, so their footprint must be carried by the rolls that **are** sold. The **functional unit** is the thing we count per: here "one sold roll".

- Footprint per sold roll = footprint per delivered roll ÷ (1 − 0.06).
- The difference is the footprint of the thrown-away rolls. We show it as the stage **Disposal**.
- **Share of a stage** = stage footprint ÷ total footprint × 100 %. The biggest stage is the **hotspot**.

We write this as a **function**: a reusable recipe with a name. You give it ingredients (here: the activities table and the returns share), it gives a result back. `def` starts the recipe, `return` hands back the result. We will use the same recipe again with sliders in Part 4.

The result is a **Series**: one column of numbers with labels (here: the five stage names).

In [ ]:
def footprint_by_stage(table, returns=RETURNS):
    """g CO2e per SOLD roll for the five stages. The thrown-away rolls appear as the stage Disposal."""
    g = table["amount"] * table["factor"] * 1000          # g CO2e per delivered roll, one value per activity
    stages = g.groupby(table["stage"]).sum()              # add up the activities of the same stage
    per_delivered = stages.sum()
    per_sold = per_delivered / (1 - returns)              # the sold rolls carry the thrown-away rolls
    stages["Disposal"] = per_sold - per_delivered         # the footprint of the thrown-away rolls
    return stages.reindex(STAGES)                         # always in the order Farming ... Disposal


roll = footprint_by_stage(activities)
per_sold = roll.sum()                                     # g CO2e per sold roll
per_delivered = per_sold - roll["Disposal"]               # g CO2e per delivered roll
shares = roll / per_sold * 100                            # % of the total, per stage
hotspot = roll.idxmax()                                   # idxmax = the label of the biggest value
annual_t = per_delivered * ROLLS_PER_YEAR / 1_000_000     # g → tonnes (1 t = 1,000,000 g)

summary = pd.DataFrame({"g CO2e per sold roll": roll.round(1), "share %": shares.round(1)})
print(summary)
print(f"\nPer delivered roll {per_delivered:.2f} g · per sold roll {per_sold:.2f} g · "
      f"per gram of roll {per_sold / ROLL_G:.2f} g · per year {annual_t:.2f} t CO2e · hotspot: {hotspot}")

In [ ]:
# CHECK Part 1
try:
    check("g CO2e per delivered roll", round(per_delivered, 2), 53.13)
    check("g CO2e per sold roll", round(per_sold, 1), 56.5, hint="per delivered ÷ 0.94")
    check("shares % (farming ... disposal)", list(shares.round(1)), [49.5, 22.5, 6.5, 15.4, 6.0])
    check("g CO2e per g of roll", round(per_sold / ROLL_G, 2), 0.94, hint="per sold roll ÷ 60 g")
    check("tonnes CO2e per year", round(annual_t, 2), 28.69, hint="per delivered roll × 540,000 ÷ 1,000,000")
    check("hotspot", hotspot, "Farming")
except Exception as e:
    not_yet(e)

### 1d · Chart 1: the roll next to the chips

A bag of Frito-Lay chips (34.5 g) causes 75 g CO₂e (Heizer et al., Supplement 5): farming 44 %, manufacture 30 %, packaging 15 %, shipping 9 %, disposal 2 %. To compare a 60 g roll with a 34.5 g bag fairly, we divide both by their weight: **g CO₂e per gram of product**.

The chart is a **horizontal stacked bar**: one bar per product, cut into the five stages. We write it as a function `draw_per_gram(ax, stages)`, because the sliders in Part 4 redraw it. `ax` is one drawing area inside a figure.

In [ ]:
FRITO_BAG_G, FRITO_TOTAL_G = 34.5, 75
frito_shares = pd.Series({"Farming": 44, "Manufacture": 30, "Packaging": 15, "Shipping": 9, "Disposal": 2})
frito = frito_shares / 100 * FRITO_TOTAL_G                 # g CO2e per bag, per stage


def draw_per_gram(ax, roll_stages, roll_g=ROLL_G):
    """Two stacked bars: g CO2e per gram of product, for the roll and for the chips."""
    products = {"Aylin's roll (60 g)": roll_stages / roll_g, "Frito-Lay chips (34.5 g)": frito / FRITO_BAG_G}
    for row, (name, per_gram) in enumerate(products.items()):
        left = 0
        for stage in STAGES:
            ax.barh(row, per_gram[stage], left=left, color=STAGE_COLOURS[stage], edgecolor="white",
                    height=0.55, label=stage if row == 0 else None)
            left += per_gram[stage]
        ax.text(left + 0.03, row, f"{left:.2f} g per g", va="center", fontweight="bold")
    ax.set_yticks([0, 1], list(products))
    ax.invert_yaxis()                                      # the roll on top
    ax.set_xlim(0, 2.6)
    ax.set_xlabel("g CO₂e per gram of product")
    ax.legend(ncol=5, frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.55))


fig, ax = plt.subplots(figsize=(10, 3.4))
draw_per_gram(ax, roll)
ax.set_title(f"Per gram, Aylin's roll causes {per_sold / ROLL_G:.2f} g CO₂e, the chips {FRITO_TOTAL_G / FRITO_BAG_G:.2f} g; "
             f"in both, farming is the biggest stage", loc="left", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### 1e · Chart 2: tonnes per year by stage

For the whole contract: g per sold roll × sold rolls per year ÷ 1,000,000 = tonnes per year. The hotspot is red. `sort_values()` sorts a list of numbers from small to large.

In [ ]:
SOLD_PER_YEAR = ROLLS_PER_YEAR * (1 - RETURNS)              # 507,600 sold rolls
tonnes = (roll * SOLD_PER_YEAR / 1_000_000).sort_values()   # tonnes CO2e per year, smallest first

fig, ax = plt.subplots(figsize=(9, 3.6))
colours = [RED if stage == hotspot else GREY for stage in tonnes.index]
bars = ax.barh(tonnes.index, tonnes, color=colours, height=0.6)
ax.bar_label(bars, labels=[f"{t:.1f} t ({t / tonnes.sum() * 100:.1f} %)" for t in tonnes], padding=4)
ax.set_xlim(0, tonnes.max() * 1.3)
ax.set_xlabel("tonnes CO₂e per year (540,000 rolls delivered)")
ax.set_title(f"{hotspot} is the hotspot: {tonnes[hotspot]:.1f} of {tonnes.sum():.2f} t CO₂e a year",
             loc="left", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

**Think (case question Q3):** Frau Lindner's rule 3 bans plastic bags. Which bar does that rule touch, and how big is it? Where is the real lever? Was your sticky-note prediction from this morning right?

*For FrischMarkt, all of Aylin's footprint is **Scope 3** (emissions of the goods it buys). For Aylin, the van is Scope 1 (her own fuel) and the electricity Scope 2 (bought energy).*

## Part 2 · Practice

Four short problems. Each one: fill in every `___`, run the cell, run the check. **Split the team:** two of you do P1 + P2, the others P3 + P4, then explain your results to each other.

### P1 · Diesel van or e-van? (case question Q4)

Jonas: *"Easy. We buy an e-van."* Is it cheaper? Does it save CO₂e? (Exhibit C)

| | Diesel van | E-van |
|---|---|---|
| Price | 38,000 € | 56,000 € |
| Energy cost per km | 10 l/100 km × 1.65 €/l = 0.165 € | 22 kWh/100 km × 0.30 €/kWh = 0.066 € |
| Other operating cost per km | 0.12 € | 0.07 € |
| km per year · life | 18,000 km · 6 years | 18,000 km · 6 years |
| Extra GHG to build (battery) | – | +8,000 kg CO₂e (ASSUMED) |

Three formulas in words:

- **Life cycle cost** (Heizer, Eq. S5-2) = purchase price + variable cost per km × km per year × years. The **variable cost per km** = energy cost per km + other operating cost per km.
- **Crossover** (km) = (price B − price A) ÷ (variable cost per km A − variable cost per km B); in years = crossover km ÷ km per year.
- **GHG break-even** (km) = extra GHG to build ÷ GHG saved per km.

**✏️ YOUR TURN:** fill in the four blanks `___`. The function's first line, `def`, is given; you write what it returns.

In [ ]:
def life_cycle_cost(price, var_cost_per_km, km_per_year, years):
    """Heizer Eq. S5-2: purchase price + variable cost per km × km per year × years."""
    return ___                                           # TODO: write the formula with the four inputs


KM_PER_YEAR, YEARS = 18_000, 6
diesel_var = 0.165 + 0.12                                # € per km: energy + other operating cost
evan_var = ___                                           # TODO: the same for the e-van

lcc_diesel = life_cycle_cost(38_000, diesel_var, KM_PER_YEAR, YEARS)
lcc_evan = ___                                           # TODO: life cycle cost of the e-van (price 56,000 €)
crossover_km = ___                                       # TODO: (price e-van − price diesel) ÷ (var diesel − var e-van)
crossover_years = crossover_km / KM_PER_YEAR

# Carbon: kg CO2e per km = energy per km × emission factor
co2_diesel_km = 10 / 100 * 2.513                         # litres per km × kg CO2e per litre
co2_evan_km = 22 / 100 * 0.363                           # kWh per km × kg CO2e per kWh (grid)
ghg_breakeven_km = 8000 / (co2_diesel_km - co2_evan_km)  # extra GHG to build ÷ GHG saved per km
t_saved_per_year = (co2_diesel_km - co2_evan_km) * KM_PER_YEAR / 1000

print(f"Life cycle cost: diesel {lcc_diesel:,.0f} €, e-van {lcc_evan:,.0f} €")
print(f"Crossover: {crossover_km:,.0f} km = {crossover_years:.2f} years (the van's life is {YEARS} years)")
print(f"GHG break-even: {ghg_breakeven_km:,.0f} km = {ghg_breakeven_km / KM_PER_YEAR:.2f} years; "
      f"saves {t_saved_per_year:.2f} t CO2e a year")

In [ ]:
# CHECK P1
try:
    check("life cycle cost diesel van (€)", round(lcc_diesel), 68780, tol=1, hint="38,000 + 0.285 × 18,000 × 6")
    check("life cycle cost e-van (€)", round(lcc_evan), 70688, tol=1, hint="56,000 + 0.136 × 18,000 × 6")
    check("crossover (km)", round(crossover_km), 120805, tol=1, hint="18,000 € ÷ 0.149 € per km")
    check("crossover (years)", round(crossover_years, 2), 6.71, hint="crossover km ÷ 18,000 km per year")
    check("GHG break-even (km)", round(ghg_breakeven_km), 46664, tol=1, hint="8,000 kg ÷ (0.2513 − 0.0799) kg per km")
    check("GHG break-even (years)", round(ghg_breakeven_km / KM_PER_YEAR, 2), 2.59)
    check("t CO2e saved per year", round(t_saved_per_year, 2), 3.09)
except Exception as e:
    not_yet(e)

**The chart (given):** total cost over the km driven, for both vans. A **numpy array** is a long list of numbers that Python can calculate with in one go: `life_cycle_cost` then gives the cost for every km value at once (with `km_per_year` = the km driven and `years` = 1). Run it after the check shows ✅.

In [ ]:
km = np.linspace(0, 160_000, 161)                        # 0, 1,000, 2,000, ... 160,000 km
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(km, life_cycle_cost(38_000, diesel_var, km, 1), color=GREY, linewidth=2.5, label="diesel van")
ax.plot(km, life_cycle_cost(56_000, evan_var, km, 1), color=GREEN, linewidth=2.5, label="e-van")
x = crossover_km
ax.plot(x, life_cycle_cost(38_000, diesel_var, x, 1), "o", color=RED, markersize=9)
ax.annotate(f"crossover {x:,.0f} km = {x / KM_PER_YEAR:.2f} years", (x, life_cycle_cost(38_000, diesel_var, x, 1)),
            xytext=(-60, 40), textcoords="offset points", color=RED, fontweight="bold", ha="right",
            arrowprops=dict(arrowstyle="-", color=RED))
life_km = KM_PER_YEAR * YEARS                            # 108,000 km in 6 years
ax.axvline(life_km, color=DARK, linestyle="--", linewidth=1)
ax.text(life_km, 30_000, f" end of life\n {life_km:,.0f} km", color=DARK)
ax.axvline(ghg_breakeven_km, color=GREEN, linestyle=":", linewidth=1.5)
ax.text(ghg_breakeven_km, 30_000, f" carbon pays back\n {ghg_breakeven_km:,.0f} km", color=GREEN)
ax.xaxis.set_major_formatter(lambda v, pos: f"{v / 1000:,.0f}k")
ax.yaxis.set_major_formatter(lambda v, pos: f"{v / 1000:,.0f}k €")
ax.set_ylim(25_000, None)
ax.set_xlabel("km driven"); ax.set_ylabel("total cost (price + km × variable cost)")
ax.set_title("The e-van pays back its carbon in under 3 years, its money not within its 6-year life",
             loc="left", fontsize=12, fontweight="bold")
ax.legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

**Think:** Jonas said *"it pays back"*. Which payback is he right about, and which not? What would have to change (diesel price, e-van price, km per year) for the money to pay back too?

### P2 · Reproduce the textbook: Blue Star (Heizer, Supplement 5, Example S2)

Blue Star needs a van for auto parts. **Ford TriVan:** 28,000 $, 24 miles per gallon, operating cost 0.20 $ per mile. **Honda CityVan** (hybrid): 32,000 $, 37 miles per gallon, 0.22 $ per mile. 22,000 miles a year, 8 years, gasoline 4.25 $ per gallon.

Fuel cost per mile = price per gallon ÷ miles per gallon. Our function `life_cycle_cost` does not care about the unit: it works with miles and $ as long as **all** inputs use miles and $.

**✏️ YOUR TURN:** fill in the three blanks.

In [ ]:
GAS, MILES_PER_YEAR, LIFE = 4.25, 22_000, 8
ford_var = GAS / 24 + 0.20                               # $ per mile: fuel + operating cost
honda_var = ___                                          # TODO: 37 miles per gallon, 0.22 $ per mile

ford = life_cycle_cost(28_000, ford_var, MILES_PER_YEAR, LIFE)
honda = ___                                              # TODO: price 32,000 $
bs_crossover_miles = ___                                 # TODO: the crossover formula from P1, in miles
bs_crossover_years = bs_crossover_miles / MILES_PER_YEAR

print(f"Ford {ford:,.0f} $ · Honda {honda:,.0f} $ · crossover {bs_crossover_miles:,.0f} miles = {bs_crossover_years:.2f} years")

In [ ]:
# CHECK P2
try:
    check("Ford TriVan life cycle cost ($)", round(ford), 94367, tol=1, hint="28,000 + (4.25/24 + 0.20) × 22,000 × 8")
    check("Honda CityVan life cycle cost ($)", round(honda), 90936, tol=1)
    check("crossover (miles)", round(bs_crossover_miles), 94745, tol=1, accept=[95012],
          hint="4,000 $ ÷ (0.3771 − 0.3349) $ per mile")
    check("crossover (years)", round(bs_crossover_years, 2), 4.31, accept=[4.32])
except Exception as e:
    not_yet(e)

**Why does the book say 95,012 miles and Python 94,745?** The book rounds the difference in cost per mile to 0.0421 $ before it divides: 4,000 ÷ 0.0421 = 95,012 miles (4.32 years). Python keeps all decimals: 0.377083… − 0.334865… = 0.042218… $, and 4,000 ÷ 0.042218… = 94,745 miles (4.31 years). Both answers are accepted. The lesson: a small rounding **in the middle** of a calculation moves the result by about 270 miles. **Round only at the end.**

### P3 · Crates and party cups: one recipe, three problems

Rule 3 of the supplier code: no plastic bags. A reusable crate costs more CO₂e to make, but saves bags on every trip.

- **Reuse break-even** (uses) = extra footprint to make the reusable item ÷ footprint saved per use. Round **up**: half a trip does not exist.
- `math.ceil(x)` rounds x **up** to the next whole number (`math.ceil(31.2)` = 32).

Crate (Exhibit D): 1.8 kg rigid plastic × 3.345 kg CO₂e per kg. One trip replaces 3 bags of 25 g LDPE (2.959 kg CO₂e per kg), and washing costs 0.03 kg CO₂e per trip.
Party cup: a reusable hard cup weighs 50 g, a single-use cup 8 g (both plastic, 3.345 g CO₂e per g); washing 10 g CO₂e per use (ASSUMED).

**✏️ YOUR TURN:** fill in the three blanks.

In [ ]:
def reuse_breakeven(extra_build, saved_per_use):
    """Uses needed before a reusable item beats the single-use one, rounded UP."""
    return ___                                           # TODO: extra_build ÷ saved_per_use, rounded up with math.ceil


crate_build = 1.8 * 3.345                                # kg CO2e to make one crate
bag = 25 / 1000 * 2.959                                  # kg CO2e per plastic bag
crate_saved_per_trip = ___                               # TODO: 3 bags saved minus washing (0.03 kg)
crate_trips = reuse_breakeven(crate_build, crate_saved_per_trip)

cup_build = 50 * 3.345                                   # g CO2e to make one hard cup
single_cup = 8 * 3.345                                   # g CO2e per single-use cup
cup_uses = ___                                           # TODO: use reuse_breakeven; washing = 10 g per use

# The same recipe for Heizer Problem S5.10 (Susan's electric car): extra GHG to build and recycle the EV
# = (30,000 + 10,000) − (14,000 + 1,000) = 25,000 lb = 400,000 oz; it saves 12 − 6 = 6 oz per mile
susan_miles = reuse_breakeven(25_000 * 16, 12 - 6)

print(f"Crate: {crate_trips} trips · cup: {cup_uses} uses · Susan's EV: {susan_miles:,} miles")

In [ ]:
# CHECK P3
try:
    check("crate break-even (trips)", crate_trips, 32, tol=0, hint="6.02 ÷ (3 × 0.074 − 0.03) = 31.4, rounded UP")
    check("party cup break-even (uses)", cup_uses, 10, tol=0, hint="167.3 ÷ (26.8 − 10), rounded UP")
    check("Susan's EV break-even (miles)", susan_miles, 66667, tol=0)
except Exception as e:
    not_yet(e)

**Think:** one crate makes 187.5 trips in its 5-year life (9,000 crate trips a year, shared by 240 crates). Is 32 trips a hard target? And the party cup: how many parties until it beats the single-use cup if you lose it after three?

### P4 · Design for disassembly: revenue retrieval (Heizer Eq. S5-1)

When a product reaches its end of life, some parts can be resold, some recycled, the rest must be disposed of. A good design earns more back.

**Revenue retrieval** = resale revenue + recycling revenue − processing cost − disposal cost (all in $ per unit, summed over the parts).

The four tables below are given (Heizer, Supplement 5: Example S1 Sound Barrier speakers, and Solved Problem S5.1 audio player). Each row is one part. `.sum()` adds up a column.

**✏️ YOUR TURN:** complete the function (one line).

In [ ]:
cols = ["resale", "recycling", "processing", "disposal"]          # $ per unit
harmonizer = pd.DataFrame([[5.93, 1.54, 3.46, 0.00], [0.00, 0.00, 4.53, 1.74], [8.56, 5.65, 6.22, 0.00],
                           [9.17, 2.65, 3.12, 0.00], [0.00, 0.00, 2.02, 1.23], [11.83, 2.10, 2.98, 0.00]],
                          columns=cols, index=["Circuit board", "Laminate back", "Coil", "Processor", "Frame", "Aluminium case"])
rocker = pd.DataFrame([[7.88, 3.54, 2.12, 0.00], [6.67, 4.56, 3.32, 0.00], [0.00, 0.00, 4.87, 1.97],
                       [8.45, 4.65, 3.43, 0.00], [0.00, 0.00, 4.65, 3.98]],
                      columns=cols, index=["Circuit board", "Coil", "Frame", "Processor", "Plastic case"])
design1 = pd.DataFrame([[4.93, 2.08, 2.98, 0.56], [0.00, 0.00, 4.12, 1.23], [6.43, 7.87, 4.73, 0.00]],
                       columns=cols, index=["Tuner", "Speaker", "Case"])
design2 = pd.DataFrame([[6.91, 4.92, 3.41, 2.13], [0.00, 0.00, 3.43, 1.97], [5.83, 3.23, 2.32, 1.57],
                        [1.67, 2.34, 4.87, 0.00]],
                       columns=cols, index=["Tuner", "Speaker", "Case", "Amplifier"])


def revenue_retrieval(parts):
    """Heizer Eq. S5-1: resale + recycling − processing − disposal, summed over all parts ($ per unit)."""
    return parts["resale"].sum() + ___                   # TODO: add recycling, subtract processing and disposal


for name, parts in [("Harmonizer", harmonizer), ("Rocker", rocker), ("Design 1", design1), ("Design 2", design2)]:
    print(f"{name:11s} revenue retrieval {revenue_retrieval(parts):6.2f} $")

In [ ]:
# CHECK P4
try:
    check("Harmonizer ($)", round(revenue_retrieval(harmonizer), 2), 22.13, tol=0.005, hint="35.49 + 11.94 − 22.33 − 2.97")
    check("Rocker ($)", round(revenue_retrieval(rocker), 2), 11.41, tol=0.005, hint="23.00 + 12.75 − 18.39 − 5.95")
    check("Solved Problem S5.1, Design 1 ($)", round(revenue_retrieval(design1), 2), 7.69, tol=0.005)
    check("Solved Problem S5.1, Design 2 ($)", round(revenue_retrieval(design2), 2), 5.20, tol=0.005)
except Exception as e:
    not_yet(e)

**Think:** which design is better for the environment, by this measure? What does the measure *not* show? (Hint: it counts dollars, not kg CO₂e.)

## Part 3 · Planning: the cheapest-first plan

This is the card game "Carbon budget" again, now in Python. Aylin can spend at most **2,000 € a year** more. The target is **−30 % = 8.61 t CO₂e a year** less (on the contract volume of 540,000 rolls).

- **Cost per tonne avoided** (€ per t CO₂e) = extra cost per year ÷ tonnes CO₂e saved per year. **Negative** = the measure saves money.
- **Cheapest-first plan:** sort the measures by € per tonne and add them from the cheapest until the target is reached.
- The picture of this is the **marginal abatement cost curve** (cost curve): one bar per measure, as **wide** as the tonnes it saves and as **high** as its € per tonne. "Abatement" means cutting emissions.
- The **EU Emissions Trading System** (EU ETS, "cap and trade") sets a price on one tonne CO₂e: on 5 Oct 2026 one allowance cost **82.4 € per tonne**. A cut that costs less per tonne than that is cheap.

Each measure is counted **alone, against today** (as on the cards). The table is given.

In [ ]:
measures = pd.DataFrame({
    "code":         ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"],
    "measure":      ["Green electricity tariff (new-build wind/solar, certified)",
                     "E-van instead of the diesel van",
                     "Reusable crates instead of plastic bags (closed loop)",
                     "Low-emission flour contract (factor 0.60 → 0.45)",
                     "Local mill 30 km away instead of 150 km",
                     "Oven heat recovery (heat for proofer + hot water)",
                     "Route planning software (15 % fewer km, as UPS does)",
                     "Halve returns (bake to FrischMarkt's daily sales data)"],
    "eur_per_year": [662, 318, -306, 1080, 432, -72, -46, 3564],      # + = extra cost, − = saves money
    "t_per_year":   [6.86, 3.09, 1.44, 3.24, 0.15, 1.18, 0.68, 0.86],   # tonnes CO2e saved per year
}).set_index("code")

TARGET = 8.61        # t CO2e per year: −30 %
BUDGET = 2000        # € per year, at most
EU_ETS = 82.4        # € per tonne CO2e, EU allowance price on 5 Oct 2026

measures

### 3a · € per tonne, sorted, with running totals

Three new tools:

- `sort_values("eur_per_t")` sorts the rows of a table by one column, smallest first.
- `cumsum()` makes a **running total** (cumulative sum): 1, 2, 3 becomes 1, 3, 6. It tells us how many tonnes we have **after** each measure.
- `.copy()` makes a copy of the table, so the original `measures` stays as it is.

**✏️ YOUR TURN:** fill in the two blanks.

In [ ]:
def cost_curve(measures):
    """The measures sorted by € per tonne (cheapest first), with running totals of tonnes and €."""
    curve = measures.copy()
    curve["eur_per_t"] = ___                                     # TODO: € per year ÷ tonnes per year (two columns)
    curve = curve.sort_values("eur_per_t")                       # cheapest first
    curve["cum_t"] = ___                                         # TODO: running total of the tonnes column
    curve["cum_eur"] = curve["eur_per_year"].cumsum()            # running total of the cost
    return curve


curve = cost_curve(measures)
curve.round(2)

### 3b · The plan: add measures until the target is reached

The rule "add from the cheapest until the target is reached" in code: keep every measure where the tonnes saved **before** it are still below the target. `curve[condition]` keeps only the rows where the condition is true (a **filter**).

**✏️ YOUR TURN:** fill in the four blanks. Jonas's plan (M2 e-van, M5 local mill, M4 low-emission flour) is picked with `.loc[...]`, which selects rows by their label.

In [ ]:
def cheapest_first(curve, target):
    """The measures of the cheapest-first plan: add them from the cheapest until the target is reached."""
    t_before = curve["cum_t"] - curve["t_per_year"]             # tonnes already saved before this measure
    return curve[t_before < target]                              # keep the rows while the target is not yet reached


plan = cheapest_first(curve, TARGET)
plan_t = ___                                                     # TODO: total tonnes of the plan (sum of a column)
plan_eur = ___                                                   # TODO: total € per year of the plan

jonas = measures.loc[["M2", "M5", "M4"]]                         # Jonas's plan: three rows by their label
jonas_t = ___                                                    # TODO
jonas_eur = ___                                                  # TODO

print(f"Cheapest-first plan {list(plan.index)}: {plan_t:.2f} t for {plan_eur:,.0f} € a year "
      f"(target {TARGET} t, budget {BUDGET:,} €)")
print(f"Jonas's plan {list(jonas.index)}: {jonas_t:.2f} t for {jonas_eur:,.0f} € a year")

*Note on € per tonne:* we divide the rounded card values, so M1 gives 662 ÷ 6.86 = 96.5 € per tonne. The spec table behind the cards was computed with all decimals (96 € per tonne). A difference of a few euros per tonne is fine: the check accepts ± 5 €, and the order of the measures is the same.

In [ ]:
# CHECK Part 3
try:
    CARD_EUR_PER_T = {"M1": 96, "M2": 103, "M3": -213, "M4": 333, "M5": 2879, "M6": -61, "M7": -67, "M8": 4141}
    check("€ per tonne, M1 ... M8 (± 5 €)", [curve.loc[c, "eur_per_t"] for c in CARD_EUR_PER_T],
          list(CARD_EUR_PER_T.values()), tol=5, hint="€ per year ÷ t per year, for every row")
    check("order, cheapest first", list(curve.index), ['M3', 'M7', 'M6', 'M1', 'M2', 'M4', 'M5', 'M8'], hint="sort_values('eur_per_t')")
    check("cheapest-first plan", list(plan.index), ['M3', 'M7', 'M6', 'M1'], hint="cum_t is the running total of t_per_year")
    check("plan: t CO2e per year", round(plan_t, 2), 10.16, hint="1.44 + 0.68 + 1.18 + 6.86")
    check("plan: € per year", round(plan_eur), 238, tol=0.5, hint="−306 − 46 − 72 + 662")
    check("plan reaches the target and stays in budget", plan_t >= TARGET and plan_eur <= BUDGET, True)
    check("Jonas: t CO2e per year", round(jonas_t, 2), 6.48)
    check("Jonas: € per year", round(jonas_eur), 1830, tol=0.5)
except Exception as e:
    not_yet(e)

### 3c · Chart 3: the cost curve

The drawing function is given. Each bar starts where the one before ended, so the x-axis shows the running total of tonnes. Bars of the plan are green, the others grey. The red dashed line is the target, the dotted line the EU ETS price. Bars that are much too tall for the chart (M5, M8) are cut off and carry their value as text.

Read the chart from left to right: the first bars **save money** (below zero). The wide green bar is M1, green electricity.

In [ ]:
def draw_mac(curve, target, highlight, title, colour=GREEN, top=500):
    """Marginal abatement cost curve: width = tonnes per year, height = € per tonne; highlight = codes to colour."""
    fig, ax = plt.subplots(figsize=(11, 5))
    left = 0
    for code, row in curve.iterrows():                           # one bar per measure, cheapest first
        width, height = row["t_per_year"], row["eur_per_t"]
        shown = min(height, top)                                 # cut off very tall bars
        ax.bar(left, shown, width=width, align="edge", color=colour if code in highlight else GREY,
               edgecolor="white", linewidth=1.5)
        middle = left + width / 2
        if height > top:                                         # write the value of a cut-off bar
            ax.text(middle, top * 0.97, f"{code} · {height:,.0f} €/t ↑", rotation=90, ha="center", va="top",
                    fontsize=9, color=DARK)
        else:
            ax.text(middle, shown + (12 if shown >= 0 else -12), code, ha="center",
                    va="bottom" if shown >= 0 else "top", fontweight="bold")
        left += width
    ax.axhline(0, color=DARK, linewidth=0.8)
    ax.axhline(EU_ETS, color=DARK, linestyle=":", linewidth=1.5)
    ax.text(0.1, EU_ETS + 10, f"EU ETS {EU_ETS:.2f} € per t", ha="left", color=DARK)
    ax.axvline(target, color=RED, linestyle="--", linewidth=1.5)
    ax.text(target + 0.1, top * 0.9, f"target {target} t", color=RED, fontweight="bold")
    ax.set_xlim(0, left * 1.01)
    ax.set_ylim(-260, top)
    ax.set_xlabel("tonnes CO₂e saved per year (running total)")
    ax.set_ylabel("€ per tonne CO₂e avoided")
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


draw_mac(curve, TARGET, list(plan.index),
         f"Cheapest first: {' + '.join(plan.index)} save {plan_t:.2f} t for {plan_eur:,.0f} € a year")

**And Jonas's plan?** The same curve, now with Jonas's three cards in orange.

In [ ]:
draw_mac(curve, TARGET, list(jonas.index),
         f"Jonas's plan: {jonas_t:.2f} t for {jonas_eur:,.0f} € a year: target missed", colour=ORANGE)

**Questions for your team** (write the answers here)

1. Jonas's plan costs 1,830 € and saves 6.48 t. The cheapest-first plan costs 238 € and saves 10.16 t. Why are Jonas's measures so expensive per tonne? (Look at the width of M5.)
2. Which measures cost less per tonne than buying an EU allowance at 82.4 €? Which are close?
3. M8 (halve the returns) saves food but costs Aylin lost sales. It is a **triple-bottom-line** conflict (people, planet, profit). Who should pay for it: Aylin, FrischMarkt, the customer?
4. The plan uses only 238 € of the 2,000 € budget. What would you do with the rest?

*Your answers:*

1.
2.
3.
4. 

## Part 4 · Optional extensions for quick teams

### E1 · Sliders: what moves the footprint of one roll?

A **widget** is a control in the notebook, such as a slider. `interact` (from `ipywidgets`) turns the inputs of a function into sliders: each time you let go of a slider, the function runs again and the chart is redrawn.

Three sliders: the **grid factor** (kg CO₂e per kWh: 0 = green electricity), the **returns share** (% of rolls thrown away) and the **flour factor** (kg CO₂e per kg flour). The function copies the activities table from Part 1, puts the slider values into it and uses `footprint_by_stage` and `draw_per_gram` again. The target is 39.6 g per sold roll (−30 %).

In [ ]:
from ipywidgets import interact, FloatSlider


def roll_per_sold(grid=0.363, returns_pct=6, flour=0.60):
    """g CO2e per sold roll, by stage, for a grid factor, a returns share (%) and a flour factor."""
    table = activities.copy()                                    # a copy: the Part 1 table stays as it is
    table.loc["Bakery electricity", "factor"] = grid
    table.loc["Wheat flour", "factor"] = flour
    return footprint_by_stage(table, returns_pct / 100)


def slider(low, high, step, start, text):
    """A slider that redraws when you let go of it (keeps Colab fast)."""
    return FloatSlider(min=low, max=high, step=step, value=start, description=text, continuous_update=False,
                       style={"description_width": "150px"}, layout=ipywidgets.Layout(width="480px"))


TARGET_G = round(per_sold * 0.7, 1)                              # −30 % per sold roll


@interact(grid=slider(0, 0.8, 0.01, 0.363, "grid kg CO₂e per kWh"),
          returns_pct=slider(0, 20, 1, 6, "returns %"),
          flour=slider(0.3, 0.9, 0.05, 0.60, "flour kg CO₂e per kg"))
def play_roll(grid, returns_pct, flour):
    stages = roll_per_sold(grid, returns_pct, flour)
    total = stages.sum()
    fig, ax = plt.subplots(figsize=(10, 3.4))
    draw_per_gram(ax, stages)
    verdict = "target reached" if total <= TARGET_G else f"{total - TARGET_G:.1f} g above the target"
    ax.set_title(f"{total:.1f} g CO₂e per sold roll (today {per_sold:.1f} g, target {TARGET_G} g): {verdict}",
                 loc="left", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

**Try:** (1) Set the grid factor to 0 (green electricity). Is the target reached with this alone? (2) Which slider moves the bar most per click? (3) Find a combination of all three sliders that reaches the target. Which real measures from the cards stand behind each slider?

### E2 · The overlap: two cards that count the same tonne

The cards count each measure **alone, against today**. But measures interact:

- **M6 heat recovery** saves electricity. Once the electricity is green (M1), saving it saves money, but **no CO₂e** any more.
- **M2 e-van:** on the grid, the e-van still causes 0.0799 kg CO₂e per km. With green electricity it causes 0, so it saves **all** of the diesel van's CO₂e: 18,000 km × 0.2513 kg = 4.52 t a year.

Below, we copy the table, change these two rows and draw the cost curve again: as it looks **after** M1. M6 is left off the new curve because it no longer saves any tonnes (Aylin keeps it anyway: it still saves 72 € a year).

*(Strictly, M8 shrinks a little too, because the thrown-away rolls no longer carry electricity CO₂e. We leave it as it is.)*

In [ ]:
green = measures.copy()
green.loc["M6", "t_per_year"] = 0                                # heat recovery now saves green power: no CO2e
green.loc["M2", "t_per_year"] = round(18_000 * 10 / 100 * 2.513 / 1000, 2)   # the e-van saves all diesel CO2e

plan_after_overlap = green.loc[list(plan.index), "t_per_year"].sum()   # the same plan, counted again
m2_green_eur_per_t = green.loc["M2", "eur_per_year"] / green.loc["M2", "t_per_year"]

print(f"M6 now saves {green.loc['M6', 't_per_year']} t; M2 now saves {green.loc['M2', 't_per_year']} t "
      f"at {m2_green_eur_per_t:.0f} € per tonne")
print(f"The cheapest-first plan really saves {plan_after_overlap:.2f} t (target {TARGET} t): "
      + ("still reached" if plan_after_overlap >= TARGET else "missed"))

curve_green = cost_curve(green.drop("M6"))                       # drop = leave a row out
draw_mac(curve_green, TARGET, ["M1", "M2"],
         f"After green power (M1), the e-van costs {m2_green_eur_per_t:.0f} € per tonne: below the EU ETS price")

In [ ]:
# CHECK E2
try:
    check("M6 after M1 (t)", green.loc["M6", "t_per_year"], 0)
    check("M2 after M1 (t)", green.loc["M2", "t_per_year"], 4.52, hint="18,000 km × 0.2513 kg ÷ 1,000")
    check("cheapest-first plan after the overlap (t)", round(plan_after_overlap, 2), 8.98,
          hint="10.16 − 1.18")
except Exception as e:
    not_yet(e)

**Think:** why is this new curve only true if Aylin buys green electricity first? What does that mean for the order in which she should decide?

### E3 · Ask your AI assistant to check a calculation

Copy this prompt into an AI assistant (ChatGPT, Claude, Gemini, Copilot …):

> *I am a first-year engineering student. Check this calculation step by step and write the unit in every line. A bakery van drives 60 km per night and uses 10 litres of diesel per 100 km. It carries 1,800 rolls per night. Diesel causes 2.513 kg CO₂e per litre (UK DESNZ conversion factors 2024). How many grams of CO₂e does the van add to one roll? State every assumption you make.*

Then judge the answer with three questions:

1. **Units:** does every line carry its unit (km, litres, kg CO₂e, g CO₂e per roll)? Is the result written as "x g CO₂e per roll"?
2. **Source of the factor:** did the AI use 2.513 kg CO₂e per litre from DESNZ 2024, or did it swap in its own number (for example 2.68 kg for pure fossil diesel)? Does it say where its number comes from?
3. **Rounding:** did it round in the middle (like the book's 0.0421 in P2)? Does its final answer match Part 1 (the `Van diesel` row) within ± 1 %?

Type the AI's final number below and run the cell.

In [ ]:
ai_answer_g = ___            # TODO: the AI's final answer, in g CO2e per roll

van_g = activities.loc["Van diesel", "g_co2e"]                   # our own result from Part 1
check("AI answer vs Part 1 (± 1 %)", ai_answer_g, round(van_g, 2), tol=van_g * 0.01,
      hint="look at its units: kg or g? per night or per roll?")

**AI reflection** (two sentences): Was the AI right? Which of the three questions caught a problem, or would have?

*Your answer:* 

## What to remember

1. **Footprint = Σ amount × emission factor.** Put the amount in the factor's unit (kWh, litres, kg, tonne-km), divide by the functional unit (one **sold** roll), and name the source of every factor.
2. **Find the hotspot first.** For Aylin it is the flour (about half the footprint). The plastic bags that the supplier code bans are only 6.5 %.
3. **Life cycle cost** and **break-even** answer different questions: the e-van pays back its carbon in under 3 years, but its money not within its 6-year life.
4. **Cut where a tonne costs least:** € per tonne, cheapest first, until the target is reached. Check for **overlap** between measures, and compare with the carbon price (EU ETS 82.4 € per tonne).

> **Sentence of the day:** *A footprint is a sum of "amount × emission factor". Find the biggest stage first, then cut where a tonne costs least — not where it looks greenest.*

---
### Sources

- Backhaus Kaya case and measures M1–M8: own case, R. Bartnik (2026). Fictional company.
- Department for Energy Security and Net Zero. (2024). *Greenhouse gas reporting: Conversion factors 2024* (condensed set, v1.1). https://assets.publishing.service.gov.uk/media/6722566a3758e4604742aa1e/ghg-conversion-factors-2024-condensed_set__for_most_users__v1_1.xlsx
- Espinoza-Orias, N., Stichnothe, H., & Azapagic, A. (2011). The carbon footprint of bread. *The International Journal of Life Cycle Assessment, 16*(4), 351–365. https://doi.org/10.1007/s11367-011-0271-0
- Heizer, J., Render, B., & Munson, C. (2020). *Operations management: Sustainability and supply chain management* (13th ed.). Pearson. Supplement 5 (Frito-Lay figure, Example S1, Example S2, Solved Problem S5.1, Problem S5.10).
- Heizer, J., Render, B., & Munson, C. (2023). *Operations management: Sustainability and supply chain management* (14th ed., Global ed.). Pearson. Read: Supplement 5, *Sustainability in the supply chain*.
- Icha, P., & Lauf, T. (2025). *Entwicklung der spezifischen Treibhausgas-Emissionen des deutschen Strommix in den Jahren 1990–2024* (Climate Change 13/2025). Umweltbundesamt. https://www.umweltbundesamt.de/publikationen/entwicklung-der-spezifischen-treibhausgas-11
- S&P Global Commodity Insights. (2026, October 5). *European carbon prices drop to 7-week low as market eyes upcoming policy agenda*. https://www.spglobal.com/energy/en/news-research/latest-news/natural-gas/100526-european-carbon-prices-drop-to-7-week-low-as-market-eyes-upcoming-policy-agenda

*Transparency note:* Claude (Anthropic) was used to draft and check this material; all figures were computed in Python; all content was reviewed by and is the responsibility of Roman Bartnik.